# RISCO DE CHURN: 3 FATORES (CONEXÃO, SUPORTE E CONTRATO)

Este notebook classifica o risco de churn de cada cliente por 3 fatores, os mais críticos, a interferência do tenure e compara o risco previsto com o churn real da history.

## Onde estão os dados
- Catálogo: `hackaton_databricks`, esquema `tabelas_hackaton`
- Origem do risco: `history_pacotes` e `inference_pacotes` (notebook Análise de Pacotes)
- Origem das tabelas de apoio e das validações 6 e 7: `history_tipada` e `inference_tipada`
- Tabelas geradas: `history_risco`, `inference_risco`, `clientes_por_servico`, `consolidado_pagamento_history`, `contrato_pagamento_history`

## Ordem das etapas
1. `history_risco` e `inference_risco`: fatores, quantidade de fatores e risco
2. Validações 1 e 2: risco x churn real e acerto da faixa Crítico (history)
3. Tabelas de apoio, com as validações 3, 4 e 5: clientes por serviço, churn por pagamento e contrato x pagamento
4. Validações 6 e 7: churn por contrato, internet e suporte; referência de 73,3% sem churn, DeviceProtection, OnlineBackup e PhoneService

## Fatores (cada um vale 1)
- **Conexão:** Fiber optic, StreamingTV ou StreamingMovies
- **Suporte:** sem TechSupport e sem OnlineSecurity (quem não tem internet não conta)
- **Contrato:** Month-to-month com Electronic check

## Risco
- `risco_churn`: 3 fatores = Crítico, 2 = Alto, 1 = Moderado, 0 = Baixo.
- `risco_churn_revisado`: aplica a trava de tenure. 3 fatores com tenure > 24 meses passam a Alto; 2 fatores com tenure <= 24 meses passam a Crítico. Tenure NULL não aciona a trava.

## Informações
- history: 5.634 clientes. inference: 1.409. Churn só existe em history, então a comparação com o churn real usa só history.
- Captura = parte do total de clientes que cancelaram (1.504) que está na faixa Crítico.
- Crítico original: 882 clientes, 63,4% cancelaram, captura 37,2% do churn.
- Crítico revisado: 1.278 clientes, 60,9% cancelaram, captura 51,7% do churn.%.

**VALIDAÇÕES**

**1. Risco x churn real (history): Crítico revisado captura 51,7% do churn, contra 37,2% do original**
- Questionado: os clientes de cada faixa de risco (risco_churn e risco_churn_revisado) cancelaram em proporções coerentes com o risco previsto?
- Resultado: cada versão soma 5.634 clientes (4.130 No e 1.504 Yes). Crítico original: 882 clientes, 559 cancelaram (63,4%), capturando 37,2% do churn total. Crítico revisado: 1.278 clientes, 778 cancelaram (60,9%), capturando 51,7%. Alto original: 1.257 clientes, 40,0% de churn. Alto revisado: 861 clientes, 33,0%. Moderado (1.889 clientes, 16,2%) e Baixo (1.606 clientes, 8,5%) são iguais nas duas versões. O churn cai a cada faixa nas duas versões.
- Decisão: usar a trava de tenure (risco_churn_revisado), com corte em 24 meses. A trava corrige o risco cliente a cliente: quem tem 3 fatores e tenure acima de 24 meses cai de Crítico para Alto (baixando o falso positivo), e quem tem 2 fatores e tenure até 24 meses sobe de Alto para Crítico (aumentando o verdadeiro positivo). Comprovado por: os cancelamentos (Yes) dentro do Crítico sobem de 559 para 778, e a captura do churn total vai de 37,2% para 51,7%; o acerto do Crítico cai pouco (63,4% para 60,9%) e o churn do Alto baixa de 40,0% para 33,0%. Cortes de 12 e 18 meses foram testados e se mostraram menos eficientes. 

**2. Acerto da faixa Crítico: revisado captura 51,7% do churn, contra 37,2% do original**
- Questionado: quanto da faixa Crítico cancelou de fato e quanto do churn total ela captura, em risco_churn e em risco_churn_revisado?
- Resultado: o Crítico original tem 882 clientes: 559 cancelaram (Yes) e 323 não (No), com 63,4% de acerto e 37,2% do churn capturado. O Crítico revisado tem 1.278 clientes: 778 Yes e 500 No, com 60,9% de acerto e 51,7% capturado.
- Decisão: a revisão amplia a faixa Crítico e eleva a captura do churn de 37,2% para 51,7%, com acerto um pouco menor no geral (63,4% para 60,9%).

**3. Clientes por serviço: PhoneService é o mais contratado (6.361) e OnlineSecurity o menos (2.019)**
- Questionado: quantos clientes contrataram cada um dos 9 serviços, em history, em inference e no total?
- Resultado: PhoneService 5.071 / 1.290 (total 6.361), InternetService 4.401 / 1.116 (5.517), MultipleLines 2.367 / 604 (2.971), StreamingMovies 2.177 / 555 (2.732), StreamingTV 2.182 / 525 (2.707), OnlineBackup 1.944 / 485 (2.429), DeviceProtection 1.932 / 490 (2.422), TechSupport 1.610 / 434 (2.044) e OnlineSecurity 1.598 / 421 (2.019) (history / inference).
- Decisão: os valores batem com o esperado (PhoneService, MultipleLines e InternetService). A tabela clientes_por_servico está validada.

**4. Churn por método de pagamento: Electronic check tem 45,1%, contra 15,6% a 19,4% dos demais**
- Questionado: o churn varia entre os métodos de pagamento?
- Resultado: Electronic check: 1.888 clientes, 852 churn (45,1%). Mailed check: 1.283 clientes, 249 churn (19,4%). Credit card (automatic): 1.234 clientes, 192 churn (15,6%). Bank transfer (automatic): 1.229 clientes, 211 churn (17,2%). Total de 5.634 clientes.
- Decisão: Electronic check concentra o maior churn, o que sustenta usá-lo como fator de risco.

**5. Contrato x pagamento: Month-to-month com Electronic check tem 53,6% de churn (1.464 clientes)**
- Questionado: o efeito do Electronic check se mantém dentro de cada tipo de contrato?
- Resultado: Month-to-month: Electronic check 53,6% (784 de 1.464), contra 31,6% a 34,1% nos demais métodos. One year: Electronic check 19,9% (58 de 291), contra 7,8% a 9,9%. Two year: Electronic check 7,5% (10 de 133), contra 1,0% a 4,1%. Electronic check é o maior churn em todos os contratos. 47,2% dos clientes Month-to-month usam Electronic check.
- Decisão: a combinação Month-to-month com Electronic check é o maior risco, e o fator de pagamento vale em qualquer contrato. Sustenta o fator Contrato do modelo.

**6. Churn total de 26,7% (1.504 de 5.634); sem TechSupport nem OnlineSecurity chega a 49,1%**
- Questionado: os números do relatório (clientes, churn e % de churn) por contrato, tipo de internet e suporte conferem, e qual o impacto financeiro?
- Resultado: total de 5.634 clientes, 1.504 churn (26,7%), receita mensal de 364.010,05 e receita mensal perdida de 112.242,90. Contrato: Month-to-month 42,7%, One year 11,6%, Two year 3,3%. Internet: Fiber optic 42,1%, DSL 19,4%, sem internet 7,2%. Suporte: sem TechSupport nem OnlineSecurity 49,1% (1.011 de 2.058), com pelo menos um 17,2% (404 de 2.343). Month-to-month responde por 96.852,45 da receita mensal perdida e Fiber optic por 92.168,90.
- Decisão: os percentuais batem com o esperado do relatório. Os três fatores (Conexão, Suporte e Contrato) correspondem aos cortes de maior churn.

**7. Referência de 73,3% sem churn; todos os 1.233 clientes sem internet têm PhoneService**
- Questionado: qual a referência de acerto se todos fossem classificados como No, qual o efeito de DeviceProtection e OnlineBackup, e todo cliente sem internet tem PhoneService?
- Resultado: sem churn em 73,3% dos 5.634 clientes. DeviceProtection: com 23,4% de churn e sem 39,0%. OnlineBackup: com 22,3% e sem 39,9%. Sem internet: 1.233 clientes, 89 churn (7,2%), todos com PhoneService = Yes.
- Decisão: a referência de 73,3% é o piso para comparar o acerto da classificação. DeviceProtection e OnlineBackup reduzem o churn, mas ficam fora dos 3 fatores. Todo cliente sem internet tem PhoneService, então PhoneService não separa esse grupo.

In [0]:
%sql
-- ============================================================
-- Criação da tabela history_risco (history)
-- ============================================================
-- O que faz: classifica o risco de churn de cada cliente de history_pacotes pela quantidade de fatores presentes (qtd_fatores, de 0 a 3), em duas versões: risco_churn (original, sem trava) e risco_churn_revisado (com trava de tenure). Em seguida, mostra o resultado ordenado por qtd_fatores para conferência.
-- Origem:    hackaton_databricks.tabelas_hackaton.history_pacotes
-- Saída:     tabela hackaton_databricks.tabelas_hackaton.history_risco (customerID, pacote_servicos, SeniorCitizen, tenure, faixa_tenure, MonthlyCharges, Churn, fator_conexao, fator_suporte, fator_contrato, qtd_fatores, fatores_presentes, risco_churn, risco_churn_revisado), 5634 linhas, uma por cliente, mais uma consulta de conferência.
-- Regras:    fator_conexao (oferta/tecnologia): InternetService = Fiber optic OU StreamingTV = Yes OU StreamingMovies = Yes. fator_suporte (desproteção): TechSupport = No E OnlineSecurity = No (quem não tem internet tem No internet service, então não conta). fator_contrato (fragilidade contratual e financeira): Contract = Month-to-month E PaymentMethod = Electronic check. risco_churn: 3 fatores = Crítico, 2 = Alto, 1 = Moderado, 0 = Baixo. risco_churn_revisado parte do risco_churn e aplica a trava: 3 fatores com tenure > 24 meses passam de Crítico para Alto (<= 24 continua Crítico), e 2 fatores com tenure <= 24 meses passam de Alto para Crítico (> 24 continua Alto). tenure NULL não aciona nenhuma trava: com 3 fatores fica Crítico e com 2 fatores fica Alto. Todo cliente cai em exatamente uma das 4 faixas, nas duas colunas. faixa_tenure é '> 24 meses', '<= 24 meses' ou 'Sem tenure' (quando tenure é NULL). fatores_presentes mostra o que no cliente gera cada fator contado em qtd_fatores, e vale "Nenhum" se qtd_fatores = 0. pacote_servicos, SeniorCitizen, tenure, MonthlyCharges e Churn são copiados de history_pacotes, sem alteração, e Churn permite comparar o risco previsto com o churn real.

CREATE OR REPLACE TABLE hackaton_databricks.tabelas_hackaton.history_risco AS
WITH fatores AS (
  SELECT
    customerID, pacote_servicos, SeniorCitizen, tenure, MonthlyCharges, Churn, InternetService, StreamingTV, StreamingMovies, TechSupport, OnlineSecurity, Contract, PaymentMethod,
    (InternetService = 'Fiber optic' OR StreamingTV = 'Yes' OR StreamingMovies = 'Yes') AS fator_conexao,
    (TechSupport = 'No' AND OnlineSecurity = 'No') AS fator_suporte,
    (Contract = 'Month-to-month' AND PaymentMethod = 'Electronic check') AS fator_contrato
  FROM hackaton_databricks.tabelas_hackaton.history_pacotes
),
contagem AS (
  SELECT *,
    (CASE WHEN fator_conexao THEN 1 ELSE 0 END
   + CASE WHEN fator_suporte THEN 1 ELSE 0 END
   + CASE WHEN fator_contrato THEN 1 ELSE 0 END) AS qtd_fatores
  FROM fatores
)
SELECT
  customerID, pacote_servicos, SeniorCitizen, tenure,
  CASE WHEN tenure IS NULL THEN 'Sem tenure'
       WHEN tenure > 24 THEN '> 24 meses'
       ELSE '<= 24 meses' END AS faixa_tenure,
  MonthlyCharges, Churn, fator_conexao, fator_suporte, fator_contrato, qtd_fatores,
  COALESCE(NULLIF(concat_ws(' | ',
    CASE WHEN fator_conexao THEN concat('Conexão: ', concat_ws(', ',
      CASE WHEN InternetService = 'Fiber optic' THEN 'Fiber optic' END,
      CASE WHEN StreamingTV = 'Yes' THEN 'StreamingTV' END,
      CASE WHEN StreamingMovies = 'Yes' THEN 'StreamingMovies' END)) END,
    CASE WHEN fator_suporte THEN 'Suporte: sem TechSupport e sem OnlineSecurity' END,
    CASE WHEN fator_contrato THEN 'Contrato: Month-to-month com Electronic check' END
  ), ''), 'Nenhum') AS fatores_presentes,
  -- risco original: apenas pela quantidade de fatores
  CASE WHEN qtd_fatores = 3 THEN 'Crítico'
       WHEN qtd_fatores = 2 THEN 'Alto'
       WHEN qtd_fatores = 1 THEN 'Moderado'
       ELSE 'Baixo' END AS risco_churn,
  -- risco revisado: quantidade de fatores + trava de tenure
  CASE WHEN qtd_fatores = 3 AND tenure > 24 THEN 'Alto'
       WHEN qtd_fatores = 3 THEN 'Crítico'
       WHEN qtd_fatores = 2 AND tenure <= 24 THEN 'Crítico'
       WHEN qtd_fatores = 2 THEN 'Alto'
       WHEN qtd_fatores = 1 THEN 'Moderado'
       ELSE 'Baixo' END AS risco_churn_revisado
FROM contagem;

SELECT customerID, pacote_servicos, SeniorCitizen, tenure, faixa_tenure, MonthlyCharges, Churn, fator_conexao, fator_suporte, fator_contrato, qtd_fatores, fatores_presentes, risco_churn, risco_churn_revisado
FROM hackaton_databricks.tabelas_hackaton.history_risco
ORDER BY qtd_fatores DESC, customerID;

customerID,pacote_servicos,SeniorCitizen,tenure,faixa_tenure,MonthlyCharges,Churn,fator_conexao,fator_suporte,fator_contrato,qtd_fatores,fatores_presentes,risco_churn,risco_churn_revisado
0011-IGKFF,"PhoneService, InternetService Fiber optic, OnlineBackup, DeviceProtection, StreamingTV, StreamingMovies",1,13,<= 24 meses,98.00,Yes,true,true,true,3,"Conexão: Fiber optic, StreamingTV, StreamingMovies | Suporte: sem TechSupport e sem OnlineSecurity | Contrato: Month-to-month com Electronic check",Crítico,Crítico
0018-NYROU,"PhoneService, InternetService Fiber optic",0,5,<= 24 meses,68.95,No,true,true,true,3,Conexão: Fiber optic | Suporte: sem TechSupport e sem OnlineSecurity | Contrato: Month-to-month com Electronic check,Crítico,Crítico
0021-IKXGC,"PhoneService, MultipleLines, InternetService Fiber optic",1,1,<= 24 meses,72.10,No,true,true,true,3,Conexão: Fiber optic | Suporte: sem TechSupport e sem OnlineSecurity | Contrato: Month-to-month com Electronic check,Crítico,Crítico
0023-XUOPT,"PhoneService, MultipleLines, InternetService Fiber optic, OnlineBackup, DeviceProtection, StreamingTV",0,13,<= 24 meses,94.10,Yes,true,true,true,3,"Conexão: Fiber optic, StreamingTV | Suporte: sem TechSupport e sem OnlineSecurity | Contrato: Month-to-month com Electronic check",Crítico,Crítico
0027-KWYKW,"PhoneService, MultipleLines, InternetService Fiber optic, StreamingTV",0,23,<= 24 meses,83.75,No,true,true,true,3,"Conexão: Fiber optic, StreamingTV | Suporte: sem TechSupport e sem OnlineSecurity | Contrato: Month-to-month com Electronic check",Crítico,Crítico
0093-EXYQL,"PhoneService, InternetService Fiber optic, StreamingTV, StreamingMovies",1,40,> 24 meses,91.55,No,true,true,true,3,"Conexão: Fiber optic, StreamingTV, StreamingMovies | Suporte: sem TechSupport e sem OnlineSecurity | Contrato: Month-to-month com Electronic check",Crítico,Alto
0094-OIFMO,"PhoneService, InternetService Fiber optic, OnlineBackup, StreamingTV, StreamingMovies",1,11,<= 24 meses,95.00,Yes,true,true,true,3,"Conexão: Fiber optic, StreamingTV, StreamingMovies | Suporte: sem TechSupport e sem OnlineSecurity | Contrato: Month-to-month com Electronic check",Crítico,Crítico
0129-KPTWJ,"PhoneService, InternetService Fiber optic, DeviceProtection, StreamingTV, StreamingMovies",0,72,> 24 meses,94.65,No,true,true,true,3,"Conexão: Fiber optic, StreamingTV, StreamingMovies | Suporte: sem TechSupport e sem OnlineSecurity | Contrato: Month-to-month com Electronic check",Crítico,Alto
0168-XZKBB,"PhoneService, MultipleLines, InternetService Fiber optic, StreamingTV",0,19,<= 24 meses,86.85,No,true,true,true,3,"Conexão: Fiber optic, StreamingTV | Suporte: sem TechSupport e sem OnlineSecurity | Contrato: Month-to-month com Electronic check",Crítico,Crítico
0187-QSXOE,"PhoneService, InternetService Fiber optic, StreamingTV, StreamingMovies",1,7,<= 24 meses,89.00,No,true,true,true,3,"Conexão: Fiber optic, StreamingTV, StreamingMovies | Suporte: sem TechSupport e sem OnlineSecurity | Contrato: Month-to-month com Electronic check",Crítico,Crítico


In [0]:
%sql
-- ============================================================
-- Criação da tabela inference_risco (inference)
-- ============================================================
-- O que faz: classifica o risco de churn de cada cliente de inference_pacotes pela quantidade de fatores presentes (qtd_fatores, de 0 a 3), em duas versões: risco_churn (original, sem trava) e risco_churn_revisado (com trava de tenure). Em seguida, mostra o resultado ordenado por qtd_fatores para conferência.
-- Origem:    hackaton_databricks.tabelas_hackaton.inference_pacotes
-- Saída:     tabela hackaton_databricks.tabelas_hackaton.inference_risco (customerID, pacote_servicos, SeniorCitizen, tenure, faixa_tenure, MonthlyCharges, fator_conexao, fator_suporte, fator_contrato, qtd_fatores, fatores_presentes, risco_churn, risco_churn_revisado), 1409 linhas, uma por cliente, mais uma consulta de conferência.
-- Regras:    fator_conexao (oferta/tecnologia): InternetService = Fiber optic OU StreamingTV = Yes OU StreamingMovies = Yes. fator_suporte (desproteção): TechSupport = No E OnlineSecurity = No (quem não tem internet tem No internet service, então não conta). fator_contrato (fragilidade contratual e financeira): Contract = Month-to-month E PaymentMethod = Electronic check. Cada fator vale 1. risco_churn: 3 fatores = Crítico, 2 = Alto, 1 = Moderado, 0 = Baixo. risco_churn_revisado parte do risco_churn e aplica a trava: 3 fatores com tenure > 24 meses passam de Crítico para Alto (<= 24 continua Crítico), e 2 fatores com tenure <= 24 meses passam de Alto para Crítico (> 24 continua Alto). tenure NULL não aciona nenhuma trava: com 3 fatores fica Crítico e com 2 fatores fica Alto. Todo cliente cai em exatamente uma das 4 faixas, nas duas colunas. faixa_tenure é '> 24 meses', '<= 24 meses' ou 'Sem tenure' (quando tenure é NULL). fatores_presentes mostra o que no cliente gera cada fator contado em qtd_fatores, e vale "Nenhum" se qtd_fatores = 0. pacote_servicos, SeniorCitizen, tenure e MonthlyCharges são copiados de inference_pacotes, sem alteração. Não há coluna Churn, pois esta base não a tem.

CREATE OR REPLACE TABLE hackaton_databricks.tabelas_hackaton.inference_risco AS
WITH fatores AS (
  SELECT
    customerID, pacote_servicos, SeniorCitizen, tenure, MonthlyCharges, InternetService, StreamingTV, StreamingMovies, TechSupport, OnlineSecurity, Contract, PaymentMethod,
    (InternetService = 'Fiber optic' OR StreamingTV = 'Yes' OR StreamingMovies = 'Yes') AS fator_conexao,
    (TechSupport = 'No' AND OnlineSecurity = 'No') AS fator_suporte,
    (Contract = 'Month-to-month' AND PaymentMethod = 'Electronic check') AS fator_contrato
  FROM hackaton_databricks.tabelas_hackaton.inference_pacotes
),
contagem AS (
  SELECT *,
    (CASE WHEN fator_conexao THEN 1 ELSE 0 END
   + CASE WHEN fator_suporte THEN 1 ELSE 0 END
   + CASE WHEN fator_contrato THEN 1 ELSE 0 END) AS qtd_fatores
  FROM fatores
)
SELECT
  customerID, pacote_servicos, SeniorCitizen, tenure,
  CASE WHEN tenure IS NULL THEN 'Sem tenure'
       WHEN tenure > 24 THEN '> 24 meses'
       ELSE '<= 24 meses' END AS faixa_tenure,
  MonthlyCharges, fator_conexao, fator_suporte, fator_contrato, qtd_fatores,
  COALESCE(NULLIF(concat_ws(' | ',
    CASE WHEN fator_conexao THEN concat('Conexão: ', concat_ws(', ',
      CASE WHEN InternetService = 'Fiber optic' THEN 'Fiber optic' END,
      CASE WHEN StreamingTV = 'Yes' THEN 'StreamingTV' END,
      CASE WHEN StreamingMovies = 'Yes' THEN 'StreamingMovies' END)) END,
    CASE WHEN fator_suporte THEN 'Suporte: sem TechSupport e sem OnlineSecurity' END,
    CASE WHEN fator_contrato THEN 'Contrato: Month-to-month com Electronic check' END
  ), ''), 'Nenhum') AS fatores_presentes,
  -- risco original: apenas pela quantidade de fatores
  CASE WHEN qtd_fatores = 3 THEN 'Crítico'
       WHEN qtd_fatores = 2 THEN 'Alto'
       WHEN qtd_fatores = 1 THEN 'Moderado'
       ELSE 'Baixo' END AS risco_churn,
  -- risco revisado: quantidade de fatores + trava de tenure
  CASE WHEN qtd_fatores = 3 AND tenure > 24 THEN 'Alto'
       WHEN qtd_fatores = 3 THEN 'Crítico'
       WHEN qtd_fatores = 2 AND tenure <= 24 THEN 'Crítico'
       WHEN qtd_fatores = 2 THEN 'Alto'
       WHEN qtd_fatores = 1 THEN 'Moderado'
       ELSE 'Baixo' END AS risco_churn_revisado
FROM contagem;

SELECT customerID, pacote_servicos, SeniorCitizen, tenure, faixa_tenure, MonthlyCharges, fator_conexao, fator_suporte, fator_contrato, qtd_fatores, fatores_presentes, risco_churn, risco_churn_revisado
FROM hackaton_databricks.tabelas_hackaton.inference_risco
ORDER BY qtd_fatores DESC, customerID;

customerID,pacote_servicos,SeniorCitizen,tenure,faixa_tenure,MonthlyCharges,fator_conexao,fator_suporte,fator_contrato,qtd_fatores,fatores_presentes,risco_churn,risco_churn_revisado
0004-TLHLJ,"PhoneService, InternetService Fiber optic, DeviceProtection",0,4,<= 24 meses,73.90,true,true,true,3,Conexão: Fiber optic | Suporte: sem TechSupport e sem OnlineSecurity | Contrato: Month-to-month com Electronic check,Crítico,Crítico
0107-YHINA,"PhoneService, MultipleLines, InternetService Fiber optic, DeviceProtection, StreamingTV, StreamingMovies",0,1,<= 24 meses,99.75,true,true,true,3,"Conexão: Fiber optic, StreamingTV, StreamingMovies | Suporte: sem TechSupport e sem OnlineSecurity | Contrato: Month-to-month com Electronic check",Crítico,Crítico
0114-PEGZZ,"PhoneService, MultipleLines, InternetService Fiber optic, OnlineBackup, DeviceProtection, StreamingTV, StreamingMovies",0,33,> 24 meses,107.55,true,true,true,3,"Conexão: Fiber optic, StreamingTV, StreamingMovies | Suporte: sem TechSupport e sem OnlineSecurity | Contrato: Month-to-month com Electronic check",Crítico,Alto
0122-OAHPZ,"PhoneService, MultipleLines, InternetService Fiber optic",0,7,<= 24 meses,73.85,true,true,true,3,Conexão: Fiber optic | Suporte: sem TechSupport e sem OnlineSecurity | Contrato: Month-to-month com Electronic check,Crítico,Crítico
0178-SZBHO,"PhoneService, MultipleLines, InternetService Fiber optic, OnlineBackup, StreamingTV",0,47,> 24 meses,87.20,true,true,true,3,"Conexão: Fiber optic, StreamingTV | Suporte: sem TechSupport e sem OnlineSecurity | Contrato: Month-to-month com Electronic check",Crítico,Alto
0225-ZORZP,"PhoneService, InternetService Fiber optic, OnlineBackup, StreamingMovies",0,3,<= 24 meses,84.30,true,true,true,3,"Conexão: Fiber optic, StreamingMovies | Suporte: sem TechSupport e sem OnlineSecurity | Contrato: Month-to-month com Electronic check",Crítico,Crítico
0259-GBZSH,"PhoneService, MultipleLines, InternetService Fiber optic, StreamingTV",0,2,<= 24 meses,85.65,true,true,true,3,"Conexão: Fiber optic, StreamingTV | Suporte: sem TechSupport e sem OnlineSecurity | Contrato: Month-to-month com Electronic check",Crítico,Crítico
0310-MVLET,"PhoneService, MultipleLines, InternetService Fiber optic, OnlineBackup, StreamingTV, StreamingMovies",0,61,> 24 meses,99.15,true,true,true,3,"Conexão: Fiber optic, StreamingTV, StreamingMovies | Suporte: sem TechSupport e sem OnlineSecurity | Contrato: Month-to-month com Electronic check",Crítico,Alto
0318-QUUOB,"PhoneService, InternetService Fiber optic, StreamingTV",0,1,<= 24 meses,80.55,true,true,true,3,"Conexão: Fiber optic, StreamingTV | Suporte: sem TechSupport e sem OnlineSecurity | Contrato: Month-to-month com Electronic check",Crítico,Crítico
0330-IVZHA,"PhoneService, InternetService Fiber optic",0,5,<= 24 meses,69.95,true,true,true,3,Conexão: Fiber optic | Suporte: sem TechSupport e sem OnlineSecurity | Contrato: Month-to-month com Electronic check,Crítico,Crítico


In [0]:
%sql
-- ============================================================
-- Validação 1: risco x churn real, original x revisado (history)
-- ============================================================
-- O que faz: conta quantos clientes de cada faixa de risco cancelaram (Churn = Yes) e quantos não (Churn = No), em risco_churn (original) e em risco_churn_revisado, para comparar o risco previsto com o churn real.
-- Origem:    history_risco
-- Saída:     consulta (tipo_risco, Churn, critico, alto, moderado, baixo, total), sem criar tabela. Esperado: 4 linhas, e cada tipo_risco soma 5634 clientes (4130 No + 1504 Yes).
-- Regras:    tipo_risco é 'risco_churn' ou 'risco_churn_revisado'. critico, alto, moderado e baixo são as quantidades de clientes em cada faixa, e total é a soma das quatro, por tipo_risco e Churn.
-- ============================================================

WITH base AS (
  SELECT Churn, 'risco_churn' AS tipo_risco, risco_churn AS faixa_risco
  FROM hackaton_databricks.tabelas_hackaton.history_risco
  UNION ALL
  SELECT Churn, 'risco_churn_revisado', risco_churn_revisado
  FROM hackaton_databricks.tabelas_hackaton.history_risco
)
SELECT
  tipo_risco,
  Churn,
  COUNT(CASE WHEN faixa_risco = 'Crítico'  THEN 1 END) AS critico,
  COUNT(CASE WHEN faixa_risco = 'Alto'     THEN 1 END) AS alto,
  COUNT(CASE WHEN faixa_risco = 'Moderado' THEN 1 END) AS moderado,
  COUNT(CASE WHEN faixa_risco = 'Baixo'    THEN 1 END) AS baixo,
  COUNT(*) AS total
FROM base
GROUP BY tipo_risco, Churn
ORDER BY tipo_risco DESC, Churn;

tipo_risco,Churn,critico,alto,moderado,baixo,total
risco_churn_revisado,No,500,577,1583,1470,4130
risco_churn_revisado,Yes,778,284,306,136,1504
risco_churn,No,323,754,1583,1470,4130
risco_churn,Yes,559,503,306,136,1504


In [0]:
%sql
-- ============================================================
-- Validação 2: acerto da faixa Crítico, original x revisado (history)
-- ============================================================
-- O que faz: mede quanto da faixa Crítico cancelou de fato e quanto do churn total ela captura, em risco_churn e em risco_churn_revisado.
-- Origem:    hackaton_databricks.tabelas_hackaton.history_risco.
-- Saída:     consulta com uma linha por tipo_risco e as colunas total_critico, critico_acertou_yes, critico_errou_no, pct_acerto_critico e pct_yes_capturados, sem criar tabela.
-- Regras:    a history_risco é empilhada (UNION ALL), uma vez para cada tipo de risco. Só conta a faixa Crítico. Acertou = Churn Yes e errou = Churn No. pct_acerto_critico = acertou ÷ total_critico. pct_yes_capturados = acertou ÷ todos os clientes com Churn Yes. Resultado ordenado por tipo_risco decrescente (risco_churn_revisado aparece primeiro).
-- ============================================================

WITH base AS (
  SELECT Churn, 'risco_churn' AS tipo_risco, risco_churn AS faixa_risco
  FROM hackaton_databricks.tabelas_hackaton.history_risco
  UNION ALL
  SELECT Churn, 'risco_churn_revisado', risco_churn_revisado
  FROM hackaton_databricks.tabelas_hackaton.history_risco
)
SELECT
  tipo_risco,
  COUNT(CASE WHEN faixa_risco = 'Crítico' THEN 1 END) AS total_critico,
  COUNT(CASE WHEN faixa_risco = 'Crítico' AND Churn = 'Yes' THEN 1 END) AS critico_acertou_yes,
  COUNT(CASE WHEN faixa_risco = 'Crítico' AND Churn = 'No' THEN 1 END) AS critico_errou_no,
  ROUND(100.0 * COUNT(CASE WHEN faixa_risco = 'Crítico' AND Churn = 'Yes' THEN 1 END)
        / COUNT(CASE WHEN faixa_risco = 'Crítico' THEN 1 END), 1) AS pct_acerto_critico,
  ROUND(100.0 * COUNT(CASE WHEN faixa_risco = 'Crítico' AND Churn = 'Yes' THEN 1 END)
        / COUNT(CASE WHEN Churn = 'Yes' THEN 1 END), 1) AS pct_yes_capturados
FROM base
GROUP BY tipo_risco
ORDER BY tipo_risco DESC;


tipo_risco,total_critico,critico_acertou_yes,critico_errou_no,pct_acerto_critico,pct_yes_capturados
risco_churn_revisado,1278,778,500,60.9,51.7
risco_churn,882,559,323,63.4,37.2


In [0]:
%sql
-- ============================================================
-- Validação 3: Clientes por serviço (history e inference)
-- ============================================================
-- O que faz: cria clientes_por_servico, com o número de clientes que contrataram cada um dos 9 serviços, em history, em inference e no total.
-- Origem:    hackaton_databricks.tabelas_hackaton.history_tipada e inference_tipada.
-- Saída:     tabela hackaton_databricks.tabelas_hackaton.clientes_por_servico (servico, clientes_history, clientes_inference, clientes_total; 9 linhas), mais consulta de conferência ordenada por clientes_total decrescente. Esperado (history / inference): PhoneService 5071 / 1290, MultipleLines 2367 / 604, InternetService 4401 / 1116.
-- Regras:    contratou = serviço com valor Yes; em InternetService, qualquer valor diferente de No (DSL ou Fiber optic). Os 9 serviços são contados em history e em inference, transformados em linhas (stack) e unidos por servico, com clientes_total = clientes_history + clientes_inference.
-- ============================================================

CREATE OR REPLACE TABLE hackaton_databricks.tabelas_hackaton.clientes_por_servico AS
WITH h AS (
  SELECT
    SUM(CASE WHEN PhoneService = 'Yes' THEN 1 ELSE 0 END) AS PhoneService,
    SUM(CASE WHEN MultipleLines = 'Yes' THEN 1 ELSE 0 END) AS MultipleLines,
    SUM(CASE WHEN InternetService <> 'No' THEN 1 ELSE 0 END) AS InternetService,
    SUM(CASE WHEN OnlineSecurity = 'Yes' THEN 1 ELSE 0 END) AS OnlineSecurity,
    SUM(CASE WHEN OnlineBackup = 'Yes' THEN 1 ELSE 0 END) AS OnlineBackup,
    SUM(CASE WHEN DeviceProtection = 'Yes' THEN 1 ELSE 0 END) AS DeviceProtection,
    SUM(CASE WHEN TechSupport = 'Yes' THEN 1 ELSE 0 END) AS TechSupport,
    SUM(CASE WHEN StreamingTV = 'Yes' THEN 1 ELSE 0 END) AS StreamingTV,
    SUM(CASE WHEN StreamingMovies = 'Yes' THEN 1 ELSE 0 END) AS StreamingMovies
  FROM hackaton_databricks.tabelas_hackaton.history_tipada
),
i AS (
  SELECT
    SUM(CASE WHEN PhoneService = 'Yes' THEN 1 ELSE 0 END) AS PhoneService,
    SUM(CASE WHEN MultipleLines = 'Yes' THEN 1 ELSE 0 END) AS MultipleLines,
    SUM(CASE WHEN InternetService <> 'No' THEN 1 ELSE 0 END) AS InternetService,
    SUM(CASE WHEN OnlineSecurity = 'Yes' THEN 1 ELSE 0 END) AS OnlineSecurity,
    SUM(CASE WHEN OnlineBackup = 'Yes' THEN 1 ELSE 0 END) AS OnlineBackup,
    SUM(CASE WHEN DeviceProtection = 'Yes' THEN 1 ELSE 0 END) AS DeviceProtection,
    SUM(CASE WHEN TechSupport = 'Yes' THEN 1 ELSE 0 END) AS TechSupport,
    SUM(CASE WHEN StreamingTV = 'Yes' THEN 1 ELSE 0 END) AS StreamingTV,
    SUM(CASE WHEN StreamingMovies = 'Yes' THEN 1 ELSE 0 END) AS StreamingMovies
  FROM hackaton_databricks.tabelas_hackaton.inference_tipada
),
hl AS (
  SELECT stack(9,
    'PhoneService', PhoneService, 'MultipleLines', MultipleLines, 'InternetService', InternetService,
    'OnlineSecurity', OnlineSecurity, 'OnlineBackup', OnlineBackup, 'DeviceProtection', DeviceProtection,
    'TechSupport', TechSupport, 'StreamingTV', StreamingTV, 'StreamingMovies', StreamingMovies
  ) AS (servico, clientes)
  FROM h
),
il AS (
  SELECT stack(9,
    'PhoneService', PhoneService, 'MultipleLines', MultipleLines, 'InternetService', InternetService,
    'OnlineSecurity', OnlineSecurity, 'OnlineBackup', OnlineBackup, 'DeviceProtection', DeviceProtection,
    'TechSupport', TechSupport, 'StreamingTV', StreamingTV, 'StreamingMovies', StreamingMovies
  ) AS (servico, clientes)
  FROM i
)
SELECT
  hl.servico,
  hl.clientes AS clientes_history,
  il.clientes AS clientes_inference,
  hl.clientes + il.clientes AS clientes_total
FROM hl
JOIN il ON hl.servico = il.servico;

SELECT * FROM hackaton_databricks.tabelas_hackaton.clientes_por_servico
ORDER BY clientes_total DESC;

servico,clientes_history,clientes_inference,clientes_total
PhoneService,5071,1290,6361
InternetService,4401,1116,5517
MultipleLines,2367,604,2971
StreamingMovies,2177,555,2732
StreamingTV,2182,525,2707
OnlineBackup,1944,485,2429
DeviceProtection,1932,490,2422
TechSupport,1610,434,2044
OnlineSecurity,1598,421,2019


In [0]:
%sql
-- ============================================================
-- Validação 4: Consolidado por método de pagamento (history)
-- ============================================================
-- O que faz: cria consolidado_pagamento_history, com uma linha por método de pagamento.
-- Origem:    hackaton_databricks.tabelas_hackaton.history_tipada.
-- Saída:     tabela hackaton_databricks.tabelas_hackaton.consolidado_pagamento_history (PaymentMethod, clientes, churn, taxa_churn_pct; 4 linhas, somando 5634 clientes), mais consulta de conferência ordenada por clientes decrescente.
-- Regras:    só history, porque o churn só existe nela. clientes = total de clientes do método. churn = clientes com Churn = Yes. taxa_churn_pct = churn ÷ clientes do método, em %, com 1 casa decimal.
-- ============================================================

CREATE OR REPLACE TABLE hackaton_databricks.tabelas_hackaton.consolidado_pagamento_history AS
SELECT
  PaymentMethod,
  COUNT(*) AS clientes,
  SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) AS churn,
  ROUND(100.0 * SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) / COUNT(*), 1) AS taxa_churn_pct
FROM hackaton_databricks.tabelas_hackaton.history_tipada
GROUP BY PaymentMethod;

SELECT * FROM hackaton_databricks.tabelas_hackaton.consolidado_pagamento_history
ORDER BY clientes DESC;

PaymentMethod,clientes,churn,taxa_churn_pct
Electronic check,1888,852,45.1
Mailed check,1283,249,19.4
Credit card (automatic),1234,192,15.6
Bank transfer (automatic),1229,211,17.2


In [0]:
%sql
-- ============================================================
-- Validação 5: Contrato x método de pagamento (history)
-- ============================================================
-- O que faz: cria contrato_pagamento_history, para ver como os clientes de cada tipo de contrato se distribuem pelos métodos de pagamento e qual o churn de cada combinação.
-- Origem:    hackaton_databricks.tabelas_hackaton.history_tipada.
-- Saída:     tabela hackaton_databricks.tabelas_hackaton.contrato_pagamento_history (Contract, PaymentMethod, clientes, pct_do_contrato, churn, taxa_churn_pct; 12 linhas, 3 contratos x 4 métodos, somando 5634 clientes), mais consulta de conferência ordenada por Contract e clientes decrescente.
-- Regras:    só history, porque o churn só existe nela. clientes = total de clientes da combinação. pct_do_contrato = % dos clientes do contrato que usam o método (cada contrato soma 100%), com 1 casa decimal. churn = clientes com Churn = Yes. taxa_churn_pct = churn ÷ clientes da combinação, em %, com 1 casa decimal.
-- ============================================================

CREATE OR REPLACE TABLE hackaton_databricks.tabelas_hackaton.contrato_pagamento_history AS
SELECT
  Contract,
  PaymentMethod,
  COUNT(*) AS clientes,
  ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY Contract), 1) AS pct_do_contrato,
  SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) AS churn,
  ROUND(100.0 * SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) / COUNT(*), 1) AS taxa_churn_pct
FROM hackaton_databricks.tabelas_hackaton.history_tipada
GROUP BY Contract, PaymentMethod;

SELECT * FROM hackaton_databricks.tabelas_hackaton.contrato_pagamento_history
ORDER BY Contract, clientes DESC;

Contract,PaymentMethod,clientes,pct_do_contrato,churn,taxa_churn_pct
Month-to-month,Electronic check,1464,47.2,784,53.6
Month-to-month,Mailed check,711,22.9,225,31.6
Month-to-month,Bank transfer (automatic),475,15.3,162,34.1
Month-to-month,Credit card (automatic),449,14.5,151,33.6
One year,Credit card (automatic),316,26.6,28,8.9
One year,Bank transfer (automatic),313,26.3,31,9.9
One year,Electronic check,291,24.5,58,19.9
One year,Mailed check,268,22.6,21,7.8
Two year,Credit card (automatic),469,34.8,13,2.8
Two year,Bank transfer (automatic),441,32.7,18,4.1


In [0]:
%sql
-- ============================================================
-- Validação 6: churn por total, contrato, tipo de internet e suporte (history)
-- ============================================================
-- O que faz: confirma os números do relatório (total de clientes, churn e % de churn) por dimensão, só em history_tipada, e mostra o impacto financeiro de cada categoria.
-- Origem:    hackaton_databricks.tabelas_hackaton.history_tipada.
-- Saída:     consulta (dimensao, categoria, clientes, churn, pct_churn, receita_mensal_total, receita_mensal_perdida; 10 linhas), sem criar tabela, ordenada por dimensao e categoria. Esperado: 5.634 clientes, 1.504 churn e 26,7% no total; contrato 42,7% / 11,6% / 3,3%; internet 42,1% (Fiber optic) e 19,4% (DSL); suporte 49,1% (1.011 de 2.058) e 17,2% (404 de 2.343).
-- Regras:    cada cliente aparece uma vez em cada dimensão (Total, Contrato, Tipo de internet e Suporte). pct_churn = churn ÷ clientes da categoria, em %, com 1 casa decimal. receita_mensal_total = soma do MonthlyCharges da categoria. receita_mensal_perdida = soma do MonthlyCharges de quem cancelou (Churn = Yes). Em Suporte, só entram clientes com internet: "Com" = TechSupport = Yes ou OnlineSecurity = Yes (pelo menos um), "Sem" = nenhum dos dois é Yes. Clientes sem internet ficam em grupo separado, fora do corte.
-- ============================================================

WITH base AS (
  SELECT Churn, MonthlyCharges, Contract, InternetService, TechSupport, OnlineSecurity
  FROM hackaton_databricks.tabelas_hackaton.history_tipada
),
longa AS (
  SELECT '1. Total' AS dimensao, 'Total' AS categoria, Churn, MonthlyCharges FROM base
  UNION ALL
  SELECT '2. Contrato', Contract, Churn, MonthlyCharges FROM base
  UNION ALL
  SELECT '3. Tipo de internet',
         CASE WHEN InternetService = 'No' THEN 'Sem internet' ELSE InternetService END,
         Churn, MonthlyCharges FROM base
  UNION ALL
  SELECT '4. Suporte',
         CASE WHEN InternetService = 'No' THEN 'Sem internet (fora do corte)'
              WHEN TechSupport = 'Yes' OR OnlineSecurity = 'Yes'
                THEN 'Com TechSupport ou OnlineSecurity'
              ELSE 'Sem TechSupport nem OnlineSecurity' END,
         Churn, MonthlyCharges FROM base
)
SELECT
  dimensao,
  categoria,
  COUNT(*) AS clientes,
  SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) AS churn,
  ROUND(100.0 * SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_churn,
  ROUND(SUM(MonthlyCharges), 2) AS receita_mensal_total,
  ROUND(SUM(CASE WHEN Churn = 'Yes' THEN MonthlyCharges ELSE 0 END), 2) AS receita_mensal_perdida
FROM longa
GROUP BY dimensao, categoria
ORDER BY dimensao, categoria;

dimensao,categoria,clientes,churn,pct_churn,receita_mensal_total,receita_mensal_perdida
1. Total,Total,5634,1504,26.7,364010.05,112242.90
2. Contrato,Month-to-month,3099,1322,42.7,205578.25,96852.45
2. Contrato,One year,1188,138,11.6,77247.00,11598.10
2. Contrato,Two year,1347,44,3.3,81184.80,3792.35
3. Tipo de internet,DSL,1924,373,19.4,111416.15,18261.00
3. Tipo de internet,Fiber optic,2477,1042,42.1,226589.15,92168.90
3. Tipo de internet,Sem internet,1233,89,7.2,26004.75,1813.00
4. Suporte,Com TechSupport ou OnlineSecurity,2343,404,17.2,184974.45,33196.25
4. Suporte,Sem TechSupport nem OnlineSecurity,2058,1011,49.1,153030.85,77233.65
4. Suporte,Sem internet (fora do corte),1233,89,7.2,26004.75,1813.00


In [0]:
%sql
-- ============================================================
-- Validação 7: % sem churn, DeviceProtection, OnlineBackup e PhoneService em quem não tem internet (history)
-- ============================================================
-- O que faz: confirma os pilares do relatório só em history_tipada: referência de 73,3% sem churn, efeito de DeviceProtection e OnlineBackup, e se todo cliente sem internet tem PhoneService.
-- Origem:    hackaton_databricks.tabelas_hackaton.history_tipada.
-- Saída:     consulta (dimensao, categoria, clientes, churn, pct_churn, pct_sem_churn, receita_mensal_total, receita_mensal_perdida; 8 linhas), sem criar tabela, ordenada por dimensao e categoria. Esperado: 14) pct_sem_churn = 73,3% | 15) DeviceProtection e OnlineBackup, com e sem | 16) 1.233 clientes sem internet, todos com PhoneService = Yes.
-- Regras:    cada cliente aparece uma vez em cada dimensão. pct_churn = churn ÷ clientes da categoria, em %, com 1 casa decimal. pct_sem_churn = 100 - pct_churn. receita_mensal_total = soma do MonthlyCharges da categoria. receita_mensal_perdida = soma do MonthlyCharges de quem cancelou (Churn = Yes). Dimensão 14: o pct_sem_churn da linha Total é a referência de acerto se todos fossem classificados como No. Dimensão 15 (a = DeviceProtection, b = OnlineBackup): "Com" = Yes, "Sem" = No, e clientes sem internet ficam em grupo separado (Sem internet). Dimensão 16: só clientes com InternetService = No, separados por PhoneService; esperado uma única linha, PhoneService = Yes, com todos os clientes sem internet.
-- ============================================================

WITH base AS (
  SELECT Churn, MonthlyCharges, InternetService, DeviceProtection, OnlineBackup, PhoneService
  FROM hackaton_databricks.tabelas_hackaton.history_tipada
),
longa AS (
  SELECT '14. Total' AS dimensao, 'Total' AS categoria, Churn, MonthlyCharges FROM base
  UNION ALL
  SELECT '15a. DeviceProtection',
         CASE WHEN DeviceProtection = 'Yes' THEN 'Com DeviceProtection'
              WHEN DeviceProtection = 'No' THEN 'Sem DeviceProtection'
              ELSE 'Sem internet' END,
         Churn, MonthlyCharges FROM base
  UNION ALL
  SELECT '15b. OnlineBackup',
         CASE WHEN OnlineBackup = 'Yes' THEN 'Com OnlineBackup'
              WHEN OnlineBackup = 'No' THEN 'Sem OnlineBackup'
              ELSE 'Sem internet' END,
         Churn, MonthlyCharges FROM base
  UNION ALL
  SELECT '16. Sem internet x PhoneService',
         CONCAT('PhoneService = ', PhoneService),
         Churn, MonthlyCharges FROM base
  WHERE InternetService = 'No'
)
SELECT
  dimensao,
  categoria,
  COUNT(*) AS clientes,
  SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) AS churn,
  ROUND(100.0 * SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_churn,
  ROUND(100.0 - 100.0 * SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_sem_churn,
  ROUND(SUM(MonthlyCharges), 2) AS receita_mensal_total,
  ROUND(SUM(CASE WHEN Churn = 'Yes' THEN MonthlyCharges ELSE 0 END), 2) AS receita_mensal_perdida
FROM longa
GROUP BY dimensao, categoria
ORDER BY dimensao, categoria;

dimensao,categoria,clientes,churn,pct_churn,pct_sem_churn,receita_mensal_total,receita_mensal_perdida
14. Total,Total,5634,1504,26.7,73.3,364010.05,112242.90
15a. DeviceProtection,Com DeviceProtection,1932,452,23.4,76.6,164186.25,40066.55
15a. DeviceProtection,Sem DeviceProtection,2469,963,39.0,61.0,173819.05,70363.35
15a. DeviceProtection,Sem internet,1233,89,7.2,92.8,26004.75,1813.00
15b. OnlineBackup,Com OnlineBackup,1944,434,22.3,77.7,161650.05,37330.40
15b. OnlineBackup,Sem OnlineBackup,2457,981,39.9,60.1,176355.25,73099.50
15b. OnlineBackup,Sem internet,1233,89,7.2,92.8,26004.75,1813.00
16. Sem internet x PhoneService,PhoneService = Yes,1233,89,7.2,92.8,26004.75,1813.00
